# Hedonic Pricing Analysis: HDB Resale Prices and Proximity to Good Schools

This notebook contains the hedonic regression workflow.

## Project objective
Estimate whether proximity to good primary schools is associated with higher HDB resale prices after controlling for:
- flat characteristics
- accessibility and amenity variables
- school density
- time and location fixed effects



In [ ]:
# ============================================================
# 1. Imports and global configuration
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

from statsmodels.tools.sm_exceptions import ValueWarning
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore", category=ValueWarning)

# Plot settings kept simple for portability
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# -----------------------------
# Project paths
# -----------------------------
DATA_DIR = Path("../data/clean")
MAIN_DATA_FILE = DATA_DIR / "final_df.csv"
ROBUST_1_FILE = DATA_DIR / "final_df_robust_1.csv"
ROBUST_2_FILE = DATA_DIR / "final_df_robust_2.csv"

# -----------------------------
# Main modelling choices
# -----------------------------
DEP_VAR = "log_real_price_psf"
MAIN_THRESHOLD = 80
MAIN_SCHOOL_VAR = f"d_0_1km_good{MAIN_THRESHOLD}"
SECONDARY_SCHOOL_VAR = f"d_1_2km_good{MAIN_THRESHOLD}"

STRUCTURAL_CONTROLS = [
    "floor_area_sqm",
    "remaining_lease_years",
]

ACCESS_CONTROLS = [
    "dist_to_cbd_km",
    "dist_to_nearest_mrt_km",
    "dist_to_nearest_mall_km",
    "dist_to_nearest_hawker_km",
    "dist_to_nearest_busstop_km",
]

SCHOOL_DENSITY_CONTROLS = [
    "countall_0_1km",
    "countall_1_2km",
]

FIXED_EFFECTS = [
    "C(town)",
    "C(year_quarter)",
    "C(storey_relative_category)",
]

THRESHOLDS_TO_TEST = [75, 80, 85, 90]
RESTRICTED_FLAT_TYPES = [4, 5]
LARGE_FLAT_TYPES = [4, 5, 6, 7]

## Helper functions

This section uses helper functions to reduce repeated code.

This improves maintainability because all major modelling logic is centralized in a few places:
- data preparation
- formula construction
- model estimation
- term extraction
- plotting

In [2]:
# ============================================================
# 2. Reusable helper functions
# ============================================================

def pct_effect(beta: float) -> float:
    """
    Convert a log-point coefficient into an exact percentage effect.

    For a semi-log model:
        % effect = (exp(beta) - 1) * 100
    """
    return (np.exp(beta) - 1) * 100


def significance_stars(p: float) -> str:
    """Return conventional significance stars for a p-value."""
    if pd.isna(p):
        return ""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""


def add_year_quarter(df: pd.DataFrame) -> pd.DataFrame:
    """Create a year-quarter string used in the fixed effects."""
    out = df.copy()
    out["year_quarter"] = (
        out["year"].astype(int).astype(str)
        + "Q"
        + out["quarter"].astype(int).astype(str)
    )
    return out


def create_model_dummies(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create flat model and flat type dummies.

    The rare 'Multi Generation' flat model is dropped from the dummy set
    because it may align too closely with a rare flat-type category and
    create perfect or near-perfect collinearity.
    """
    out = df.copy()

    flat_model_dummies = pd.get_dummies(out["flat_model"], prefix="fm", drop_first=True)
    if "fm_Multi Generation" in flat_model_dummies.columns:
        flat_model_dummies = flat_model_dummies.drop(columns=["fm_Multi Generation"])

    flat_type_dummies = pd.get_dummies(out["flat_type"], prefix="ft", drop_first=True)

    out = pd.concat([out, flat_model_dummies, flat_type_dummies], axis=1)
    return out


def get_dummy_columns(df: pd.DataFrame):
    """Return sorted lists of generated flat model and flat type dummies."""
    flat_model_vars = sorted([c for c in df.columns if c.startswith("fm_")])
    flat_type_vars = sorted([c for c in df.columns if c.startswith("ft_")])
    return flat_model_vars, flat_type_vars


def q_wrap(var_name: str) -> str:
    """Wrap names with spaces or hyphens for statsmodels formulas."""
    return f'Q("{var_name}")' if (" " in var_name or "-" in var_name) else var_name


def check_required_columns(df: pd.DataFrame, required_cols: list):
    """Raise a clear error if required variables are missing."""
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def build_formula(
    dep_var: str,
    school_vars: list,
    structural_controls: list,
    access_controls: list = None,
    school_density_controls: list = None,
    flat_model_vars: list = None,
    flat_type_vars: list = None,
    fixed_effects: list = None,
    extra_terms: list = None,
) -> str:
    """
    Build a regression formula dynamically to avoid duplicated long strings.
    """
    rhs_terms = []
    rhs_terms += school_vars
    rhs_terms += structural_controls

    if access_controls:
        rhs_terms += access_controls
    if school_density_controls:
        rhs_terms += school_density_controls
    if flat_model_vars:
        rhs_terms += [q_wrap(v) for v in flat_model_vars]
    if flat_type_vars:
        rhs_terms += flat_type_vars
    if fixed_effects:
        rhs_terms += fixed_effects
    if extra_terms:
        rhs_terms += extra_terms

    return f"{dep_var} ~ {' + '.join(rhs_terms)}"


def run_hedonic_model(
    df: pd.DataFrame,
    formula: str,
    cov_type: str = "HC3",
    cluster_groups=None,
):
    """
    Estimate an OLS hedonic regression with robust or clustered standard errors.
    """
    if cov_type == "cluster":
        if cluster_groups is None:
            raise ValueError("cluster_groups must be provided when cov_type='cluster'")
        return smf.ols(formula=formula, data=df).fit(
            cov_type="cluster",
            cov_kwds={"groups": cluster_groups}
        )
    return smf.ols(formula=formula, data=df).fit(cov_type=cov_type)


def summarize_terms(model, terms: list, model_name: str) -> pd.DataFrame:
    """
    Extract coefficients and convert them into stakeholder-friendly summary rows.
    """
    rows = []
    for term in terms:
        beta = model.params.get(term, np.nan)
        se = model.bse.get(term, np.nan)
        p = model.pvalues.get(term, np.nan)

        rows.append({
            "model": model_name,
            "term": term,
            "coef": beta,
            "std_err": se,
            "p_value": p,
            "pct_effect": pct_effect(beta) if pd.notnull(beta) else np.nan,
            "stars": significance_stars(p),
        })
    return pd.DataFrame(rows)


def add_ci_columns(summary_df: pd.DataFrame) -> pd.DataFrame:
    """
    Add 95% confidence intervals both in coefficient space and percentage-effect space.
    """
    out = summary_df.copy()
    out["ci_low_coef"] = out["coef"] - 1.96 * out["std_err"]
    out["ci_high_coef"] = out["coef"] + 1.96 * out["std_err"]
    out["ci_low_pct"] = (np.exp(out["ci_low_coef"]) - 1) * 100
    out["ci_high_pct"] = (np.exp(out["ci_high_coef"]) - 1) * 100
    return out


def make_term_label(term: str) -> str:
    """Create cleaner labels for charts and stakeholder tables."""
    label_map = {
        "d_0_1km_good75": "0-1 km from good school (p75)",
        "d_1_2km_good75": "1-2 km from good school (p75)",
        "d_0_1km_good80": "0-1 km from good school (p80)",
        "d_1_2km_good80": "1-2 km from good school (p80)",
        "d_0_1km_good85": "0-1 km from good school (p85)",
        "d_1_2km_good85": "1-2 km from good school (p85)",
        "d_0_1km_good90": "0-1 km from good school (p90)",
        "d_1_2km_good90": "1-2 km from good school (p90)",
        "count_0_1km_good80": "Count of good schools within 0-1 km",
        "count_1_2km_good80": "Count of good schools within 1-2 km",
        "dist_nearest_goodschool_80": "Distance to nearest good school",
        "dist_nearest_goodschool_80_sq": "Distance to nearest good school squared",
    }
    return label_map.get(term, term)


def compute_vif(df: pd.DataFrame, vars_to_check: list) -> pd.DataFrame:
    """
    Compute VIF for selected regressors.

    Fixed effects are excluded on purpose because the diagnostic here is meant
    to evaluate the main structural, treatment, and location variables.
    """
    X = df[vars_to_check].dropna().copy()
    X = sm.add_constant(X)
    return pd.DataFrame({
        "variable": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    })


def coefficient_dotwhisker(df_plot: pd.DataFrame, x_col: str, low_col: str, high_col: str, y_col: str, title: str, xlabel: str):
    """Simple coefficient plot used repeatedly for stakeholder-facing artifacts."""
    plt.figure(figsize=(10, max(4, len(df_plot) * 0.55)))
    plt.axvline(x=0, linestyle="--", linewidth=1)
    for i, row in df_plot.reset_index(drop=True).iterrows():
        plt.errorbar(
            x=row[x_col],
            y=i,
            xerr=[[row[x_col] - row[low_col]], [row[high_col] - row[x_col]]],
            fmt="o"
        )
    plt.yticks(range(len(df_plot)), df_plot[y_col])
    plt.xlabel(xlabel)
    plt.ylabel("")
    plt.title(title)
    plt.tight_layout()
    plt.show()

## Load and prepare the main estimation sample

This section does three things:
1. loads the main analytical dataset
2. creates dummies and fixed-effect labels
3. checks that all required variables exist before estimation


In [3]:
# ============================================================
# 3. Load and prepare main dataset
# ============================================================

df_raw = pd.read_csv(MAIN_DATA_FILE)
df_raw = add_year_quarter(df_raw)

print("Raw data shape:", df_raw.shape)
print("Number of towns:", df_raw["town"].nunique())
print("Number of year-quarter periods:", df_raw["year_quarter"].nunique())

# Basic data checks related to potentially problematic rare categories
print("\nUnique flat types for 'Multi Generation':")
print(df_raw.loc[df_raw["flat_model"] == "Multi Generation", "flat_type"].unique())

df_model = create_model_dummies(df_raw)
flat_model_vars, flat_type_vars = get_dummy_columns(df_model)

main_required_cols = (
    [DEP_VAR, MAIN_SCHOOL_VAR, SECONDARY_SCHOOL_VAR]
    + STRUCTURAL_CONTROLS
    + ACCESS_CONTROLS
    + SCHOOL_DENSITY_CONTROLS
    + ["town", "year_quarter", "storey_relative_category"]
)
check_required_columns(df_model, main_required_cols)

reg_df = df_model.dropna(subset=main_required_cols).copy()

print("\nEstimation sample shape:", reg_df.shape)
print("Share within 0-1 km of good school:", reg_df[MAIN_SCHOOL_VAR].mean())
print("Share within 1-2 km of good school:", reg_df[SECONDARY_SCHOOL_VAR].mean())

Raw data shape: (291419, 51)
Number of towns: 26
Number of year-quarter periods: 52

Unique flat types for 'Multi Generation':
[7]

Estimation sample shape: (291419, 76)
Share within 0-1 km of good school: 0.4226011344490236
Share within 1-2 km of good school: 0.5794200103630854


## Treatment summary

Before running regressions, it is useful to show how common the treatment groups are in the estimation sample. This helps stakeholders understand whether the estimated premiums are based on broad or narrow subsamples.

In [4]:
treatment_summary = pd.DataFrame({
    "group": [MAIN_SCHOOL_VAR, SECONDARY_SCHOOL_VAR],
    "share_in_sample": [
        reg_df[MAIN_SCHOOL_VAR].mean(),
        reg_df[SECONDARY_SCHOOL_VAR].mean(),
    ],
    "count": [
        reg_df[MAIN_SCHOOL_VAR].sum(),
        reg_df[SECONDARY_SCHOOL_VAR].sum(),
    ],
})
treatment_summary["label"] = treatment_summary["group"].map(make_term_label)
treatment_summary

,group,share_in_sample,count,label
0,d_0_1km_good80,0.422601,123154,0-1 km from good school (p80)
1,d_1_2km_good80,0.579420,168854,1-2 km from good school (p80)


## Model setup

The analysis uses a sequential hedonic framework.

- Model 1 includes the treatment variables, structural controls, flat-type controls, flat-model controls, and fixed effects.
- Model 2 adds accessibility controls to test whether the estimated school premium partly captures broader locational advantages.
- Model 3 adds school-density controls and is treated as the preferred specification because it better isolates the premium linked to proximity to a high-demand school rather than proximity to schools in general.

In [5]:
# ============================================================
# 4. Main hedonic specifications
# ============================================================

school_terms = [MAIN_SCHOOL_VAR, SECONDARY_SCHOOL_VAR]

formula_m1 = build_formula(
    dep_var=DEP_VAR,
    school_vars=school_terms,
    structural_controls=STRUCTURAL_CONTROLS,
    flat_model_vars=flat_model_vars,
    flat_type_vars=flat_type_vars,
    fixed_effects=FIXED_EFFECTS,
)

formula_m2 = build_formula(
    dep_var=DEP_VAR,
    school_vars=school_terms,
    structural_controls=STRUCTURAL_CONTROLS,
    access_controls=ACCESS_CONTROLS,
    flat_model_vars=flat_model_vars,
    flat_type_vars=flat_type_vars,
    fixed_effects=FIXED_EFFECTS,
)

formula_m3 = build_formula(
    dep_var=DEP_VAR,
    school_vars=school_terms,
    structural_controls=STRUCTURAL_CONTROLS,
    access_controls=ACCESS_CONTROLS,
    school_density_controls=SCHOOL_DENSITY_CONTROLS,
    flat_model_vars=flat_model_vars,
    flat_type_vars=flat_type_vars,
    fixed_effects=FIXED_EFFECTS,
)

model1 = run_hedonic_model(reg_df, formula_m1, cov_type="HC3")
model2 = run_hedonic_model(reg_df, formula_m2, cov_type="HC3")
model3 = run_hedonic_model(reg_df, formula_m3, cov_type="HC3")

summary_main = pd.concat([
    summarize_terms(model1, school_terms, "Model 1"),
    summarize_terms(model2, school_terms, "Model 2"),
    summarize_terms(model3, school_terms, "Model 3"),
], ignore_index=True)

summary_main["label"] = summary_main["term"].map(make_term_label)
summary_main = add_ci_columns(summary_main)

summary_main

,model,term,coef,std_err,p_value,pct_effect,stars,label,ci_low_coef,ci_high_coef,ci_low_pct,ci_high_pct
0,Model 1,d_0_1km_good80,0.007242,0.000436,6.955717e-62,0.726826,***,0-1 km from good school (p80),0.006387,0.008097,0.640734,0.812991
1,Model 1,d_1_2km_good80,0.016418,0.000464,6.334101e-274,1.655401,***,1-2 km from good school (p80),0.015508,0.017328,1.562937,1.747949
2,Model 2,d_0_1km_good80,0.008688,0.000397,5.681175e-106,0.872609,***,0-1 km from good school (p80),0.007909,0.009467,0.794076,0.951204
3,Model 2,d_1_2km_good80,0.007517,0.000421,2.925324e-71,0.754503,***,1-2 km from good school (p80),0.006691,0.008342,0.671375,0.837699
4,Model 3,d_0_1km_good80,0.011532,0.000408,1.146776e-175,1.159864,***,0-1 km from good school (p80),0.010732,0.012332,1.078981,1.240811
5,Model 3,d_1_2km_good80,0.004139,0.000433,1.117381e-21,0.414770,***,1-2 km from good school (p80),0.003291,0.004987,0.329642,0.499970


## Preferred interpretation

The table below translates the preferred specification into plain-English interpretation. This is useful for policy or stakeholder communication because the coefficient is expressed in percentage terms rather than only log points.

In [6]:
preferred_terms = summarize_terms(model3, school_terms, "Preferred model")
preferred_terms["label"] = preferred_terms["term"].map(make_term_label)
preferred_terms = add_ci_columns(preferred_terms)

preferred_terms["interpretation"] = preferred_terms.apply(
    lambda row: (
        f"{row['label']} is associated with an estimated {row['pct_effect']:.2f}% "
        f"difference in HDB resale price psf in the preferred specification."
    ),
    axis=1,
)

preferred_terms[[
    "model", "label", "coef", "std_err", "p_value", "pct_effect",
    "ci_low_pct", "ci_high_pct", "stars", "interpretation"
]]

,model,label,coef,std_err,p_value,pct_effect,ci_low_pct,ci_high_pct,stars,interpretation
0,Preferred model,0-1 km from good school (p80),0.011532,0.000408,1.146776e-175,1.159864,1.078981,1.240811,***,0-1 km from good school (p80) is associated wi...
1,Preferred model,1-2 km from good school (p80),0.004139,0.000433,1.117381e-21,0.414770,0.329642,0.499970,***,1-2 km from good school (p80) is associated wi...


## Sensitivity check: clustered standard errors by town

The baseline uses HC3 robust standard errors. This section compares those results with standard errors clustered by town. The purpose is not to replace the baseline automatically, but to test whether inference changes materially once within-town dependence is allowed for.

In [7]:
model3_cluster = run_hedonic_model(
    reg_df,
    formula_m3,
    cov_type="cluster",
    cluster_groups=reg_df["town"]
)

cluster_compare = pd.DataFrame({
    "term": school_terms,
    "label": [make_term_label(t) for t in school_terms],
    "coef_hc3": [model3.params[t] for t in school_terms],
    "se_hc3": [model3.bse[t] for t in school_terms],
    "p_hc3": [model3.pvalues[t] for t in school_terms],
    "coef_cluster_town": [model3_cluster.params[t] for t in school_terms],
    "se_cluster_town": [model3_cluster.bse[t] for t in school_terms],
    "p_cluster_town": [model3_cluster.pvalues[t] for t in school_terms],
})

cluster_compare

,term,label,coef_hc3,se_hc3,p_hc3,coef_cluster_town,se_cluster_town,p_cluster_town
0,d_0_1km_good80,0-1 km from good school (p80),0.011532,0.000408,1.146776e-175,0.011532,0.007737,0.136089
1,d_1_2km_good80,1-2 km from good school (p80),0.004139,0.000433,1.117381e-21,0.004139,0.007569,0.584487


## Multicollinearity diagnostics

This section checks whether the main treatment and continuous control variables are excessively collinear. Fixed effects are excluded from the VIF diagnostic because the goal here is to assess the substantive regressors rather than the large set of categorical controls.

In [8]:
vif_vars = [
    MAIN_SCHOOL_VAR,
    SECONDARY_SCHOOL_VAR,
    "countall_0_1km",
    "countall_1_2km",
    "floor_area_sqm",
    "remaining_lease_years",
    "mature_estate",
    "dist_to_cbd_km",
    "dist_to_nearest_mrt_km",
    "dist_to_nearest_mall_km",
    "dist_to_nearest_hawker_km",
    "dist_to_nearest_busstop_km",
]

vif_results = compute_vif(reg_df, vif_vars).sort_values("VIF", ascending=False)
vif_results

,variable,VIF
0,const,106.632703
7,mature_estate,3.582270
8,dist_to_cbd_km,3.308562
3,countall_0_1km,1.527524
4,countall_1_2km,1.405357
2,d_1_2km_good80,1.261323
11,dist_to_nearest_hawker_km,1.250633
6,remaining_lease_years,1.248354
9,dist_to_nearest_mrt_km,1.234124
10,dist_to_nearest_mall_km,1.208285


## Robustness 1: alternative good-school thresholds

This section tests whether the estimated premium depends heavily on the exact percentile cutoff used to define a good school. If the main findings are stable across thresholds, the results are less likely to be an artefact of one arbitrary classification choice.

In [9]:
def run_threshold_model(df: pd.DataFrame, threshold: int, flat_model_vars: list, flat_type_vars: list) -> pd.DataFrame:
    main_var = f"d_0_1km_good{threshold}"
    sec_var = f"d_1_2km_good{threshold}"

    required_cols = (
        [DEP_VAR, main_var, sec_var]
        + STRUCTURAL_CONTROLS
        + ACCESS_CONTROLS
        + SCHOOL_DENSITY_CONTROLS
        + ["town", "year_quarter", "storey_relative_category"]
    )
    check_required_columns(df, required_cols)

    sub_df = df.dropna(subset=required_cols).copy()

    formula = build_formula(
        dep_var=DEP_VAR,
        school_vars=[main_var, sec_var],
        structural_controls=STRUCTURAL_CONTROLS,
        access_controls=ACCESS_CONTROLS,
        school_density_controls=SCHOOL_DENSITY_CONTROLS,
        flat_model_vars=flat_model_vars,
        flat_type_vars=flat_type_vars,
        fixed_effects=FIXED_EFFECTS,
    )

    model = run_hedonic_model(sub_df, formula, cov_type="HC3")
    out = summarize_terms(model, [main_var, sec_var], f"Threshold {threshold}")
    out["threshold"] = threshold
    out["label"] = out["term"].map(make_term_label)
    return add_ci_columns(out)

threshold_results = pd.concat(
    [run_threshold_model(reg_df, t, flat_model_vars, flat_type_vars) for t in THRESHOLDS_TO_TEST],
    ignore_index=True
)

threshold_results[[
    "model", "threshold", "label", "coef", "std_err", "p_value",
    "pct_effect", "ci_low_pct", "ci_high_pct", "stars"
]]

,model,threshold,label,coef,std_err,p_value,pct_effect,ci_low_pct,ci_high_pct,stars
0,Threshold 75,75,0-1 km from good school (p75),0.004587,0.000400,1.925415e-30,0.459745,0.381016,0.538536,***
1,Threshold 75,75,1-2 km from good school (p75),0.007974,0.000460,2.996297e-67,0.800566,0.709681,0.891532,***
2,Threshold 80,80,0-1 km from good school (p80),0.011532,0.000408,1.146776e-175,1.159864,1.078981,1.240811,***
3,Threshold 80,80,1-2 km from good school (p80),0.004139,0.000433,1.117381e-21,0.414770,0.329642,0.499970,***
4,Threshold 85,85,0-1 km from good school (p85),0.012607,0.000446,1.256350e-175,1.268637,1.180115,1.357236,***
5,Threshold 85,85,1-2 km from good school (p85),0.003350,0.000439,2.237813e-14,0.335586,0.249342,0.421903,***
6,Threshold 90,90,0-1 km from good school (p90),0.006662,0.000498,9.539747e-41,0.668389,0.570098,0.766776,***
7,Threshold 90,90,1-2 km from good school (p90),0.001243,0.000454,6.200159e-03,0.124358,0.035288,0.213507,***


## Robustness 2: count of good schools instead of a dummy

The main model asks whether being inside a specific distance band of at least one good school is associated with higher prices. This robustness check instead uses the number of good schools in each band. It tests whether the result reflects intensity of school concentration rather than only binary proximity.

In [10]:
count_main_var = "count_0_1km_good80"
count_sec_var = "count_1_2km_good80"

count_required_cols = (
    [DEP_VAR, count_main_var, count_sec_var]
    + STRUCTURAL_CONTROLS
    + ACCESS_CONTROLS
    + SCHOOL_DENSITY_CONTROLS
    + ["town", "year_quarter", "storey_relative_category"]
)
check_required_columns(reg_df, count_required_cols)

formula_count = build_formula(
    dep_var=DEP_VAR,
    school_vars=[count_main_var, count_sec_var],
    structural_controls=STRUCTURAL_CONTROLS,
    access_controls=ACCESS_CONTROLS,
    school_density_controls=SCHOOL_DENSITY_CONTROLS,
    flat_model_vars=flat_model_vars,
    flat_type_vars=flat_type_vars,
    fixed_effects=FIXED_EFFECTS,
)

model_count = run_hedonic_model(reg_df.dropna(subset=count_required_cols), formula_count, cov_type="HC3")
count_results = summarize_terms(model_count, [count_main_var, count_sec_var], "Count-based model")
count_results["label"] = count_results["term"].map(make_term_label)
count_results = add_ci_columns(count_results)
count_results

,model,term,coef,std_err,p_value,pct_effect,stars,label,ci_low_coef,ci_high_coef,ci_low_pct,ci_high_pct
0,Count-based model,count_0_1km_good80,0.007803,0.000355,5.435339e-107,0.783357,***,Count of good schools within 0-1 km,0.007107,0.008499,0.713227,0.853536
1,Count-based model,count_1_2km_good80,0.000729,0.000270,6.843643e-03,0.072918,***,Count of good schools within 1-2 km,0.000201,0.001257,0.020065,0.125800


## Robustness 3: quadratic amenity-distance controls

This section allows accessibility variables to enter nonlinearly. The aim is to test whether the estimated school premium is sensitive to a more flexible specification of general locational controls.

In [11]:
reg_df_quad = reg_df.copy()

quad_base_vars = [
    "dist_to_cbd_km",
    "dist_to_nearest_mrt_km",
    "dist_to_nearest_mall_km",
    "dist_to_nearest_hawker_km",
    "dist_to_nearest_busstop_km",
]

for v in quad_base_vars:
    reg_df_quad[f"{v}_sq"] = reg_df_quad[v] ** 2

formula_quad = build_formula(
    dep_var=DEP_VAR,
    school_vars=school_terms,
    structural_controls=STRUCTURAL_CONTROLS,
    access_controls=ACCESS_CONTROLS,
    school_density_controls=SCHOOL_DENSITY_CONTROLS,
    flat_model_vars=flat_model_vars,
    flat_type_vars=flat_type_vars,
    fixed_effects=FIXED_EFFECTS,
    extra_terms=[f"{v}_sq" for v in quad_base_vars],
)

quad_required_cols = main_required_cols + [f"{v}_sq" for v in quad_base_vars]
model_quad = run_hedonic_model(reg_df_quad.dropna(subset=quad_required_cols), formula_quad, cov_type="HC3")
quad_results = summarize_terms(model_quad, school_terms, "Quadratic accessibility controls")
quad_results["label"] = quad_results["term"].map(make_term_label)
quad_results = add_ci_columns(quad_results)
quad_results

,model,term,coef,std_err,p_value,pct_effect,stars,label,ci_low_coef,ci_high_coef,ci_low_pct,ci_high_pct
0,Quadratic accessibility controls,d_0_1km_good80,0.014797,0.000417,6.082922e-276,1.490721,***,0-1 km from good school (p80),0.013980,0.015614,1.407826,1.573684
1,Quadratic accessibility controls,d_1_2km_good80,0.006142,0.000437,8.684317e-45,0.616060,***,1-2 km from good school (p80),0.005284,0.006999,0.529839,0.702355


## Robustness 4: alternative GSI weights

If you created additional datasets using alternative weighting schemes for the good-school index, this section lets you test whether the proximity premium is robust to those alternative definitions.

The code assumes the alternative datasets use the same variable structure as the main estimation file.

In [12]:
robust_weight_files = [
    ("Alternative weights 1 (0.7SDI_0.15GEP_0.15SAP)", ROBUST_1_FILE),
    ("Alternative weights 2 (0.8SDI_0.1GEP_0.1SAP)", ROBUST_2_FILE),
]

processed_weight_results = []

for model_name, file_path in robust_weight_files:
    if file_path.exists():
        temp = pd.read_csv(file_path)
        temp = add_year_quarter(temp)
        temp = create_model_dummies(temp)
        temp_flat_model_vars, temp_flat_type_vars = get_dummy_columns(temp)

        req = (
            [DEP_VAR, MAIN_SCHOOL_VAR, SECONDARY_SCHOOL_VAR]
            + STRUCTURAL_CONTROLS
            + ACCESS_CONTROLS
            + SCHOOL_DENSITY_CONTROLS
            + ["town", "year_quarter", "storey_relative_category"]
        )
        check_required_columns(temp, req)
        temp_reg = temp.dropna(subset=req).copy()

        temp_formula = build_formula(
            dep_var=DEP_VAR,
            school_vars=school_terms,
            structural_controls=STRUCTURAL_CONTROLS,
            access_controls=ACCESS_CONTROLS,
            school_density_controls=SCHOOL_DENSITY_CONTROLS,
            flat_model_vars=temp_flat_model_vars,
            flat_type_vars=temp_flat_type_vars,
            fixed_effects=FIXED_EFFECTS,
        )

        temp_model = run_hedonic_model(temp_reg, temp_formula, cov_type="HC3")
        temp_result = summarize_terms(temp_model, school_terms, model_name)
        temp_result["label"] = temp_result["term"].map(make_term_label)
        processed_weight_results.append(temp_result)

if processed_weight_results:
    alt_weight_results = add_ci_columns(pd.concat(processed_weight_results, ignore_index=True))
else:
    print("Alternative weight files were not found. Update the file paths if needed.")

alt_weight_results

,model,term,coef,std_err,p_value,pct_effect,stars,label,ci_low_coef,ci_high_coef,ci_low_pct,ci_high_pct
0,Alternative weights 1 (0.7SDI_0.15GEP_0.15SAP),d_0_1km_good80,0.011620,0.000404,8.200183e-182,1.168760,***,0-1 km from good school (p80),0.010828,0.012412,1.088659,1.248925
1,Alternative weights 1 (0.7SDI_0.15GEP_0.15SAP),d_1_2km_good80,0.004951,0.000431,1.521652e-30,0.496376,***,1-2 km from good school (p80),0.004107,0.005796,0.411512,0.581313
2,Alternative weights 2 (0.8SDI_0.1GEP_0.1SAP),d_0_1km_good80,0.010835,0.000398,1.855831e-163,1.089424,***,0-1 km from good school (p80),0.010056,0.011615,1.010659,1.168250
3,Alternative weights 2 (0.8SDI_0.1GEP_0.1SAP),d_1_2km_good80,0.004923,0.000430,2.231146e-30,0.493464,***,1-2 km from good school (p80),0.004080,0.005765,0.408855,0.578145


## Heterogeneity 1: larger flats

This interaction checks whether the estimated school premium is stronger for larger flats, which are more likely to be occupied by households with children.

In [13]:
hetero_large_df = reg_df.copy()
hetero_large_df["large_flat"] = hetero_large_df["flat_type"].isin(LARGE_FLAT_TYPES).astype(int)
hetero_large_df["d_0_1km_good80_large"] = hetero_large_df[MAIN_SCHOOL_VAR] * hetero_large_df["large_flat"]
hetero_large_df["d_1_2km_good80_large"] = hetero_large_df[SECONDARY_SCHOOL_VAR] * hetero_large_df["large_flat"]

formula_large = build_formula(
    dep_var=DEP_VAR,
    school_vars=[
        MAIN_SCHOOL_VAR,
        SECONDARY_SCHOOL_VAR,
        "d_0_1km_good80_large",
        "d_1_2km_good80_large",
    ],
    structural_controls=STRUCTURAL_CONTROLS,
    access_controls=ACCESS_CONTROLS,
    school_density_controls=SCHOOL_DENSITY_CONTROLS,
    flat_model_vars=flat_model_vars,
    flat_type_vars=flat_type_vars,
    fixed_effects=FIXED_EFFECTS,
)

model_large = run_hedonic_model(hetero_large_df, formula_large, cov_type="HC3")
print(model_large.summary())

                            OLS Regression Results                            
Dep. Variable:     log_real_price_psf   R-squared:                       0.824
Model:                            OLS   Adj. R-squared:                  0.824
Method:                 Least Squares   F-statistic:                 1.227e+04
Date:                Wed, 08 Apr 2026   Prob (F-statistic):               0.00
Time:                        16:05:58   Log-Likelihood:             2.8311e+05
No. Observations:              291419   AIC:                        -5.660e+05
Df Residuals:                  291302   BIC:                        -5.647e+05
Df Model:                         116                                         
Covariance Type:                  HC3                                         
                                                   coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------

## Heterogeneity 2: mature versus non-mature estates

This interaction checks whether the school premium differs between mature and non-mature estates. This helps assess whether the estimated effect is stronger in already established locations.

### Using Interaction

In [14]:
# ============================================================
# Mature-estate heterogeneity via interaction terms
# ============================================================

hetero_mature_df = reg_df.copy()

# Ensure binary integer
hetero_mature_df["mature_estate"] = hetero_mature_df["mature_estate"].astype(int)

# Interaction terms
hetero_mature_df[f"{MAIN_SCHOOL_VAR}_mature"] = (
    hetero_mature_df[MAIN_SCHOOL_VAR] * hetero_mature_df["mature_estate"]
)
hetero_mature_df[f"{SECONDARY_SCHOOL_VAR}_mature"] = (
    hetero_mature_df[SECONDARY_SCHOOL_VAR] * hetero_mature_df["mature_estate"]
)

mature_interaction_terms = [
    MAIN_SCHOOL_VAR,
    SECONDARY_SCHOOL_VAR,
    f"{MAIN_SCHOOL_VAR}_mature",
    f"{SECONDARY_SCHOOL_VAR}_mature",
]

formula_mature_interaction = build_formula(
    dep_var=DEP_VAR,
    school_vars=mature_interaction_terms,
    structural_controls=STRUCTURAL_CONTROLS,
    access_controls=ACCESS_CONTROLS,
    school_density_controls=SCHOOL_DENSITY_CONTROLS,
    flat_model_vars=flat_model_vars,
    flat_type_vars=flat_type_vars,
    fixed_effects=FIXED_EFFECTS,
)

model_mature_interaction = run_hedonic_model(
    hetero_mature_df,
    formula_mature_interaction,
    cov_type="HC3"
)

print(model_mature_interaction.summary())

                            OLS Regression Results                            
Dep. Variable:     log_real_price_psf   R-squared:                       0.823
Model:                            OLS   Adj. R-squared:                  0.823
Method:                 Least Squares   F-statistic:                 1.223e+04
Date:                Wed, 08 Apr 2026   Prob (F-statistic):               0.00
Time:                        16:06:18   Log-Likelihood:             2.8221e+05
No. Observations:              291419   AIC:                        -5.642e+05
Df Residuals:                  291302   BIC:                        -5.630e+05
Df Model:                         116                                         
Covariance Type:                  HC3                                         
                                                   coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------

In [15]:
# ============================================================
# Summarise mature-estate interaction results
# ============================================================

interaction_results_mature = summarize_terms(
    model_mature_interaction,
    mature_interaction_terms,
    "Mature-estate interaction model"
)

interaction_results_mature["label"] = interaction_results_mature["term"].replace({
    MAIN_SCHOOL_VAR: "0-1 km premium (non-mature estates)",
    SECONDARY_SCHOOL_VAR: "1-2 km premium (non-mature estates)",
    f"{MAIN_SCHOOL_VAR}_mature": "Extra 0-1 km premium in mature estates",
    f"{SECONDARY_SCHOOL_VAR}_mature": "Extra 1-2 km premium in mature estates",
})

interaction_results_mature = add_ci_columns(interaction_results_mature)

interaction_results_mature[[
    "model", "term", "label", "coef", "std_err", "p_value",
    "pct_effect", "ci_low_pct", "ci_high_pct", "stars"
]]

,model,term,label,coef,std_err,p_value,pct_effect,ci_low_pct,ci_high_pct,stars
0,Mature-estate interaction model,d_0_1km_good80,0-1 km premium (non-mature estates),0.011371,0.000447,7.920640e-143,1.143605,1.055053,1.232234,***
1,Mature-estate interaction model,d_1_2km_good80,1-2 km premium (non-mature estates),0.001682,0.000459,2.449851e-04,0.168300,0.078320,0.258361,***
2,Mature-estate interaction model,d_0_1km_good80_mature,Extra 0-1 km premium in mature estates,0.001044,0.001064,3.262263e-01,0.104482,-0.104002,0.313400,
3,Mature-estate interaction model,d_1_2km_good80_mature,Extra 1-2 km premium in mature estates,0.010917,0.001182,2.635384e-20,1.097703,0.863673,1.332276,***


In [16]:
# ============================================================
# Compute implied premiums for mature vs non-mature estates
# ============================================================

b_non_01 = model_mature_interaction.params[MAIN_SCHOOL_VAR]
b_non_12 = model_mature_interaction.params[SECONDARY_SCHOOL_VAR]

b_diff_01 = model_mature_interaction.params[f"{MAIN_SCHOOL_VAR}_mature"]
b_diff_12 = model_mature_interaction.params[f"{SECONDARY_SCHOOL_VAR}_mature"]

b_mat_01 = b_non_01 + b_diff_01
b_mat_12 = b_non_12 + b_diff_12

premium_compare = pd.DataFrame({
    "group": ["Non-mature estates", "Mature estates"],
    "0-1 km log coef": [b_non_01, b_mat_01],
    "0-1 km pct premium": [pct_effect(b_non_01), pct_effect(b_mat_01)],
    "1-2 km log coef": [b_non_12, b_mat_12],
    "1-2 km pct premium": [pct_effect(b_non_12), pct_effect(b_mat_12)],
})

premium_compare

,group,0-1 km log coef,0-1 km pct premium,1-2 km log coef,1-2 km pct premium
0,Non-mature estates,0.011371,1.143605,0.001682,0.16830
1,Mature estates,0.012415,1.249281,0.012599,1.26785


## Final project summary

This notebook directly addresses the project objective in three ways:

1. **Core estimation**  
   The main hedonic regressions estimate whether flats within 0-1 km and 1-2 km of good schools command different resale prices after controlling for housing and locational factors.

2. **Robustness**  
   The threshold, count-based, quadratic-control and alternative-weight checks test whether the result is stable to different modelling choices.

3. **Heterogeneity**  
   Additional analysis was conducted across households and space to give users a deeper understanding of the non-uniformity of the effects of good primary school

